# Predicting Transfer Market Value in Football using Football Manager 2024 Data
### MSc Artificial Intelligence and Data Science – Osasumwen Okunkpolor

## Summary: This notebook cleans and engineers player data from Football Manager 2024 to create structured, numeric features for modelling football player market values.

### importing libraries

In [1]:
import re
import numpy as np
import pandas as pd
from dateutil import parser as dateparser

### Configuration: Define file paths, constants, and the FM24 reference snapshot date.

In [2]:
# configuration
INPUT_PATH  = "../data/fm_with_uefa.xlsx"   # file with UEFA coefficients merged
OUTPUT_CSV  = "../data/fm_cleaned_ready.csv"
OUTPUT_XLSX = "../data/fm_cleaned_ready.xlsx"
CONTRACT_REF_DATE = pd.Timestamp("2023-07-01")  # FM24 snapshot date

### Helper Functions for Monetary Parsing, Physical Attributes,Date and Contract Parsing,Position Simplification

In [3]:
# HELPERS 
_MULTIPLIERS = {"M": 1e6, "K": 1e3, "": 1}

In [4]:
# Helper to convert FM-style money strings to float 
# Converts FM-style money strings (e.g. "£50M", "£100K") into numeric float values.
def parse_money(value):
    """
    Converts Football Manager money strings like:
    '£50M', '£100K', '£100,000', or 'Not Set' into numeric float values.
    Returns np.nan for anything unparseable.
    """
    if not isinstance(value, str):
        return value  # already numeric or NaN
    # Remove currency symbols and commas    
    s = value.strip().replace(",", "").replace("£", "").replace("€", "")
     # Handle blanks or placeholder entries
    if s == "" or s.lower() in ["-", "not set", "n/a", "none"]:
        return np.nan
     # Extract numeric and optional magnitude suffix (K or M)
    match = re.match(r"([\d.]+)\s*([MK]?)", s, flags=re.I)
    if not match:
        return np.nan
    num, suffix = match.groups()
    num = float(num)
     # Apply multiplier based on suffix
    mult = 1e6 if suffix.upper() == "M" else 1e3 if suffix.upper() == "K" else 1
    return num * mult

# Parses FM transfer-value ranges (e.g "£50M–£65M") and flags "Not for Sale" cases
def parse_transfer_value(value):
    """Parse Transfer Value ranges and Not for Sale"""
    if not isinstance(value, str):
        return np.nan, np.nan, np.nan, 0
    s = value.strip()
    # Detect players explicitly marked as Not for Sale
    if s.lower().startswith("not for sale"):
        return np.nan, np.nan, np.nan, 1
     # Extract all numeric values within the range    
    parts = re.findall(r"£\s*([\d,.]+)\s*([MK]?)", s, flags=re.I)
    if not parts:
        return np.nan, np.nan, np.nan, 0
    # Convert extracted parts into numeric values    
    vals = [float(p[0].replace(",","")) * _MULTIPLIERS.get(p[1].upper(),1) for p in parts]
     # Compute min, max, and midpoint values
    vmin, vmax = min(vals), max(vals)
    vmid = (vmin + vmax) / 2
    return vmin, vmax, vmid, 0


# Extracts numeric height in cm from text strings (e.g. "185 cm")
def parse_height_cm(x):
    if isinstance(x, str) and "cm" in x:
        return float(x.split("cm")[0].strip())
    return pd.to_numeric(x, errors="coerce")


# Extracts numeric weight in kg from text strings (e.g. "80 kg")
def parse_weight_kg(x):
    if isinstance(x, str) and "kg" in x:
        return float(x.split("kg")[0].strip())
    return pd.to_numeric(x, errors="coerce")
    

# Converts text-based date fields into pandas datetime objects
def parse_date(d):
    if pd.isna(d) or d is None or str(d).strip() == "":
        return pd.NaT
    try:
        # Use dateparser for flexible parsing of mixed date formats
        return dateparser.parse(str(d), dayfirst=True, fuzzy=True)
    except:
        return pd.NaT


# Groups detailed FM position labels into two broad categories
def coarse_position_group(text):
    if isinstance(text, str) and "GK" in text:
        return "GK"
    return "Outfield"


# Converts weekly wage strings (e.g. "£250K p/w") to numeric per-week values
def parse_wage(value):
    """Convert Football Manager wage strings (e.g. '£250K p/w') to numeric per-week value."""
    if not isinstance(value, str):
        return value
     # Clean and normalise text    
    s = value.replace("£", "").replace(",", "").replace("p/w", "").replace(" ", "").upper()
    # Extract numeric component and optional suffix
    match = re.match(r"([\d.]+)\s*([MK]?)", s)
    if not match:
        return np.nan
    num, suffix = match.groups()
    # Apply suffix multiplier
    mult = 1e6 if suffix == "M" else 1e3 if suffix == "K" else 1
    return float(num) * mult

print("Functions created successfully")

Functions created successfully


### Data Cleaning and Normalisation: Loads the FM24 dataset, parses key fields (transfer values, contracts, wages),  normalises attributes, and saves a cleaned, structured dataset for modelling.


In [5]:
# Load the Football Manager 2024 dataset from the input file
df = pd.read_excel(INPUT_PATH)

# TRANSFER VALUE
# Parse transfer-value ranges and extract minimum, maximum, midpoint, and "Not for Sale" flag
parsed = df["Transfer Value"].apply(parse_transfer_value)
df[["fm_value_min","fm_value_max","fm_value_mid","is_not_for_sale"]] = pd.DataFrame(parsed.tolist(), index=df.index)

# HEIGHT/WEIGHT
# Convert player height and weight strings to numeric centimetre and kilogram values
df["height_cm"] = df["Height"].apply(parse_height_cm)
df["weight_kg"] = df["Weight"].apply(parse_weight_kg)

# CONTRACT
# Parse contract start and expiry dates
df["begins_dt"]  = df["Begins"].apply(parse_date)
df["expires_dt"] = df["Expires"].apply(parse_date)

# Calculate total contract length in years
df["contract_years_total"] = (df["expires_dt"] - df["begins_dt"]).dt.days / 365.25

# Define the FM24 snapshot date for contract calculations
SNAPSHOT = pd.Timestamp("2023-07-01")  # FM24 reference date

# Calculate remaining contract duration relative to the snapshot date
df["contract_years_remaining_raw"] = (df["expires_dt"] - SNAPSHOT).dt.days / 365.25

# Flag expired contracts
df["is_expired_contract"] = (df["contract_years_remaining_raw"] < 0).astype(int)
# Clip negative durations to zero
df["contract_years_remaining"] = df["contract_years_remaining_raw"].clip(lower=0)


# ENGINEERED FEATURES
# Compute player development potential
df["potential_growth"] = df["PA"] - df["CA"]

# Categorise players into position groups (e.g., GK, Outfield)
df["position_group"] = df["Position"].apply(coarse_position_group)

# RELEASE CLAUSES
# Parse release-clause values from text and convert to numeric
df["release_all"]      = df["Min Fee Rls"].apply(parse_money)
df["release_releg"]    = df["Relegation Release"].apply(parse_money)
df["release_domestic"] = df["Min Fee Rls to Domestic Clubs"].apply(parse_money)

# Create binary indicators for players with release clauses
df["has_release_all"]      = df["release_all"].notna().astype(int)
df["has_release_releg"]    = df["release_releg"].notna().astype(int)
df["has_release_domestic"] = df["release_domestic"].notna().astype(int)


# NORMALISATION
# Normalise ability and potential attributes to a 0–1 scale
df["CA_norm"] = df["CA"] / 200
df["PA_norm"] = df["PA"] / 200
df["potential_growth_norm"] = (df["PA"] - df["CA"]) / 200

# Normalise club and world reputation scores to a 0–1 scale
df["CR_norm"] = df["CR"] / 10000
df["WR_norm"] = df["WR"] / 10000

# Normalise injury proneness
if "Inj Pr" in df.columns:
    df["inj_pr_norm"] = df["Inj Pr"] / 20
elif "Injury Pr" in df.columns:
    df["inj_pr_norm"] = df["Injury Pr"] / 20

# Parse weekly wage values and apply logarithmic transformation to reduce skew
df["Wage_clean"] = df["Wage"].apply(parse_wage)
df["Wage_log"] = np.log1p(df["Wage_clean"])

# TARGET
# Define the modelling target variable as the midpoint of market value range
df["y"] = df["fm_value_mid"]


# Save the cleaned dataset to CSV and Excel formats for later use
df.to_csv(OUTPUT_CSV, index=False)
df.to_excel(OUTPUT_XLSX, index=False, sheet_name="cleaned")

print(" Dataset cleaned and saved with features + target ready for modelling")

 Dataset cleaned and saved with features + target ready for modelling


### League Strength Construction:  Calculates a composite league_strength variable by combining UEFA coefficients with average player market values per league. Each country’s top division serves as a benchmark, and lower divisions are scaled proportionally to reflect competitive quality.

In [6]:
# Load the preprocessed dataset containing player data and UEFA country points
df = pd.read_excel("../data/fm_cleaned_ready.xlsx")

# Step 1: Average FM value per (country, division)
# Compute the mean market value for each league within every country
league_means = (
    df.groupby(["country","Division"])["fm_value_mid"]
      .mean()
      .reset_index()
      .rename(columns={"fm_value_mid":"league_avg_value"})
)
# Step 2: Top league average per country 
# Determine the highest league average to serve as a national benchmark
league_means["top_league_avg"] = (
    league_means.groupby("country")["league_avg_value"].transform("max")
)

# Step 3: Compute intra-country scaling factors
# Express each league’s average as a ratio of its country’s top division
league_means["scaler"] = league_means["league_avg_value"] / league_means["top_league_avg"]


# Step 4: Merge scaling factors back into the player dataset
# Add each player’s league scaling value
df = df.merge(
    league_means[["country","Division","scaler"]],
    on=["country","Division"], how="left"
)

# Step 5: Compute final league_strength
# Combine UEFA coefficients with intra-country scaling to capture competitive quality
df["league_strength"] = df["uefa_points"] * df["scaler"]

# Clean up missing entries
# Remove records without valid league_strength values
df = df[df["league_strength"].notna()].copy()

# Save the final dataset with league-strength values

df.to_excel("../data/fm_with_league_strength.xlsx", index=False)
df.to_csv("../data/fm_with_league_strength.csv", index=False)

print("Added scaled league_strength to dataset")

Added scaled league_strength to dataset
